In [9]:
library(Seurat)
library(arrow)
library(ggplot2)
library(tibble)
library(future)
plan("multicore", workers = 24)
options(jupyter.plot_scale=1.5)
options(future.globals.maxSize = 8 * 1024^3)

In [10]:
base.path <- "/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/Working_project/MB/scRNA_reference"

In [17]:
reference_data_MB_All <- readRDS(paste0(base.path, "/SHH_Riemondy/MB_All/MB_All.rds"))
reference_data_MB_SHH <- readRDS(paste0(base.path, "/SHH_Riemondy/MB_SHH/MB_SHH.rds"))

In [18]:
reference_data_MB_All$cell_id <- rownames(reference_data_MB_All)
reference_data_MB_SHH$cell_id <- rownames(reference_data_MB_SHH)
reference_data_MB_All@meta.data$label <- reference_data_MB_All@meta.data$cell_type
reference_data_MB_All@meta.data$label[reference_data_MB_All@meta.data$cell_id %in% reference_data_MB_SHH@meta.data$cell_id] = reference_data_MB_SHH@meta.data$Subpopulations
Idents(object = reference_data_MB_All) <- "subgroup"
reference_data_SHH_sub <- subset(x = reference_data_MB_All, idents = c("SHH"))
Idents(object = reference_data_SHH_sub) <- "label"
reference_data <- subset(x = reference_data_SHH_sub, idents = c("malignant", "lymphocytes"), invert = TRUE)


In [19]:
reference_data <- SCTransform(reference_data, assay = "RNA", variable.features.n = nrow(reference_data))
reference_data <- RunPCA(reference_data, npcs = 30, features = rownames(reference_data))

Running SCTransform on assay: RNA

Running SCTransform on layer: counts

vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

Variance stabilizing transformation of count matrix of size 20271 by 6075

Model formula is y ~ log_umi

Get Negative Binomial regression parameters per gene

Using 2000 genes, 5000 cells

There are 5 estimated thetas smaller than 1e-07 - will be set to 1e-07

Found 3 outliers - those will be ignored in fitting/regularization step


Second step: Get residuals using fitted parameters for 20271 genes

Computing corrected count matrix for 20271 genes

Calculating gene attributes

Wall clock passed: Time difference of 1.052967 mins

Determine variable features

Centering data matrix

Getting residuals for block 1(of 2) for counts dataset

Getting residuals for block 2(of 2) for counts dataset

Centering data matrix

Finished calculating residuals for counts

Set default assay to SCT

PC_ 1 
Positive:  HLA-DRA, CD74, APOE, HLA-DPA1, HLA-DPB

Assay (v5) data with 26841 features for 6075 cells
First 10 features:
 MIR1302-2HG, OR4F5, AL627309.1, AL627309.3, AL732372.1, AL669831.2,
AL669831.5, FAM87B, LINC00115, FAM41C 
Layers:
 counts 

In [20]:
xenium_file = c("B59460","Ctrl_1","Ctrl_2","P38685","P52407","P6055125","P62560","Treated_1","Treated_2")

In [21]:
for (i in 1:length(xenium_file)){
    file_name <- xenium_file[i]
    library_id <- file_name
    xenium.obj <- LoadXenium(paste0("/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/Working_project/MB/Xenium_Brain/Xenium_RAW/",file_name))
    xenium.obj <- subset(xenium.obj, subset = nCount_Xenium > 0)
    xenium.obj <- SCTransform(xenium.obj, assay = "Xenium", variable.features.n = nrow(xenium.obj))
    xenium.obj <- RunPCA(xenium.obj, npcs = 30, features = rownames(xenium.obj))
    ref.anchors <- FindTransferAnchors(reference = reference_data, query = xenium.obj, dims = 1:30, normalization.method = "SCT")
    predictions <- TransferData(anchorset = ref.anchors, refdata = reference_data$label, dims = 1:30)
    xenium.obj <- AddMetaData(xenium.obj, metadata = predictions)
    df <- tibble::rownames_to_column(xenium.obj@meta.data, "cell_id")
    arrow::write_parquet(df, paste0("/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/Working_project/MB/Xenium_Brain/Xenium_RAW/",file_name,"/",library_id,"_lt_meta.parquet"))
}

10X data contains more than one type and is being returned as a list containing matrices of each type.

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warn

In [49]:
reference_data

An object of class Seurat 
26841 features across 6075 samples within 1 assay 
Active assay: RNA (26841 features, 0 variable features)
 1 layer present: counts

In [37]:
table(reference_data_MB_All@meta.data$label)


                      lymphocytes              macrophage_monocytes 
                              713                              1715 
                        malignant oligodendrocytes_astrocytes_other 
                            25159                               271 
                           SHH-A1                            SHH-A2 
                             1580                               500 
                           SHH-B1                            SHH-B2 
                             3714                              1584 
                           SHH-C1                            SHH-C2 
                             3474                              1204 
                           SHH-X1 
                               32 

In [26]:
rownames(reference_data_MB_SHH@meta.data)

[1] "X801_AAACCCAGTGCAAGAC"  "X801_AAAGAACCACGTAGTT" 
   [3] "X801_AAAGGATCAAGATGTA"  "X801_AAAGGATCAGAGCGTA" 
   [5] "X801_AAATGGACACGTCATA"  "X801_AACAAAGGTACGGTTT" 
   [7] "X801_AACAAAGTCGGCCTTT"  "X801_AACAACCGTACCGTCG" 
   [9] "X801_AACAACCGTTGACTAC"  "X801_AACAACCTCTAGACCA" 
  [11] "X801_AACAAGACACATGACT"  "X801_AACAAGATCCGATGTA" 
  [13] "X801_AACACACTCCAGTGCG"  "X801_AACCACATCGATTGGT" 
  [15] "X801_AACCCAACAAGATTGA"  "X801_AACCCAAGTCCTGGTG" 
  [17] "X801_AACCTGAGTACTTGTG"  "X801_AACCTTTGTTAATCGC" 
  [19] "X801_AACGGGAAGCGCAATG"  "X801_AACTTCTGTACGCTTA" 
  [21] "X801_AAGAACAAGTCTACCA"  "X801_AAGACAAAGGCTAGCA" 
  [23] "X801_AAGACTCAGGGCGAGA"  "X801_AAGCATCGTTGCCAAT" 
  [25] "X801_AAGCCATGTTGCACGC"  "X801_AAGCGTTCAAGGCTTT" 
  [27] "X801_AAGCGTTCACTGTCCT"  "X801_AAGGAATAGGTCCAGA" 
  [29] "X801_AAGTACCGTAGGAGGG"  "X801_AAGTTCGAGCGTGAGT" 
  [31] "X801_AATCACGGTTCCATTT"  "X801_AATCACGTCACGAGGA" 
  [33] "X801_AATCGTGAGCGATTCT"  "X801_AATCGTGCACTCCTGT" 
  [35] "X801_AATGACCCAATAAGGT"  "X801_AATGACCGTCAATGGG" 
  [37] "X801_AATGCCAAGCCTCAGC"  "X801_AATGCCAAGGAAGAAC" 
  [39] "X801_AATTCCTGTACCCACG"  "X801_AATTTCCCACAAACGG" 
  [41] "X801_AATTTCCTCAGAGTGG"  "X801_ACAAAGAAGCGCTGAA" 
  [43] "X801_ACAAGCTAGTCGTTAC"  "X801_ACAAGCTCAAGCACAG" 
  [45] "X801_ACACAGTTCCCATGGG"  "X801_ACACGCGTCATCGGGC" 
  [47] "X801_ACAGAAACAATTGCAC"  "X801_ACATCCCGTTCGAAGG" 
  [49] "X801_ACATCGATCATTCGTT"  "X801_ACCAACATCTTCGCTG" 
  [51] "X801_ACCACAAGTAAGATTG"  "X801_ACCCTCAAGTGCTACT" 
  [53] "X801_ACCCTTGAGAACTTCC"  "X801_ACCCTTGTCGTAGCTA" 
  [55] "X801_ACCCTTGTCTATCGGA"  "X801_ACCGTTCCAGAGGCTA" 
  [57] "X801_ACCTACCAGACTACCT"  "X801_ACCTACCAGGGACACT" 
  [59] "X801_ACCTACCTCCACTGAA"  "X801_ACCTGTCAGTTGCCTA" 
  [61] "X801_ACCTGTCGTTCAACGT"  "X801_ACGATCAAGCAAACAT" 
  [63] "X801_ACGATCAAGCACCGTC"  "X801_ACGATGTAGTGGTTGG" 
  [65] "X801_ACGCACGCAGGGAGAG"  "X801_ACGCACGTCATTGTGG" 
  [67] "X801_ACGCACGTCGGTTCAA"  "X801_ACGGAAGCAGGGTCTC" 
  [69] "X801_ACGGAAGGTGAATTGA"  "X801_ACGGAAGTCGCTACGG" 
  [71] "X801_ACGGGTCAGTAACGAT"  "X801_ACGGTCGCAGAGCGTA" 
  [73] "X801_ACGGTCGGTACTTCCC"  "X801_ACGGTCGGTAGCACAG" 
  [75] "X801_ACGGTTATCTGGGCAC"  "X801_ACGTCCTCATCTTCGC" 
  [77] "X801_ACGTTCCTCTGTCAGA"  "X801_ACTATCTCACGCTGCA" 
  [79] "X801_ACTATCTGTTGCCAAT"  "X801_ACTATTCAGACAGTCG" 
  [81] "X801_ACTATTCCATCTCGTC"  "X801_ACTATTCGTCTACACA" 
  [83] "X801_ACTCCCAAGCTGCGAA"  "X801_ACTCCCAGTAACTTCG" 
  [85] "X801_ACTCTCGAGAAGGTAG"  "X801_ACTCTCGCAACCCGCA" 
  [87] "X801_ACTGATGGTAGTATAG"  "X801_ACTGATGTCTAGACAC" 
  [89] "X801_ACTGCAAAGAGCCATG"  "X801_ACTGCAAGTTCGAACT" 
  [91] "X801_ACTGTCCAGGCCACCT"  "X801_ACTGTCCTCTGAGGCC" 
  [93] "X801_ACTGTGAAGCTCGGCT"  "X801_ACTGTGAGTGTTGATC" 
  [95] "X801_ACTGTGATCTGCTCTG"  "X801_ACTTATCCATACTGAC" 
  [97] "X801_ACTTATCTCTCTGCTG"  "X801_ACTTCCGAGTGCTACT" 
  [99] "X801_ACTTCGCCATCTGCGG"  "X801_ACTTTCAGTCTTGAAC" 
 [101] "X801_AGAAGCGAGGGCTTCC"  "X801_AGAAGTACAGAGTCAG" 
 [103] "X801_AGAAGTACAGTGTACT"  "X801_AGACAAAGTTTCGTAG" 
 [105] "X801_AGACCATAGTCGCCCA"  "X801_AGACCATCAAATCGGG" 
 [107] "X801_AGACCATGTCTTCTAT"  "X801_AGACCATTCCGGGACT" 
 [109] "X801_AGACTCACACGTAGTT"  "X801_AGAGAATAGCGTTCCG" 
 [111] "X801_AGAGAATCAAGAGGCT"  "X801_AGAGAATCAATCACGT" 
 [113] "X801_AGAGAGCCATTCGGGC"  "X801_AGAGAGCTCCCGAGGT" 
 [115] "X801_AGAGCAGCAGAGTTGG"  "X801_AGAGCCCCAACCGCTG" 
 [117] "X801_AGAGCCCCATACTTTC"  "X801_AGAGCCCGTCCACTTC" 
 [119] "X801_AGAGCCCGTCGCTTAA"  "X801_AGATAGACATCGTGGC" 
 [121] "X801_AGATAGATCATGCGGC"  "X801_AGATCCACACAGGATG" 
 [123] "X801_AGATCCAGTACTAGCT"  "X801_AGATCGTAGTTGCCTA" 
 [125] "X801_AGATGCTAGGGTGAGG"  "X801_AGCATCAAGTCATAGA" 
 [127] "X801_AGCCAATAGGTGAGCT"  "X801_AGCCAGCTCCGTTGGG" 
 [129] "X801_AGCTCAAAGAATCGTA"  "X801_AGCTCAACATGGCGCT" 
 [131] "X801_AGGACTTGTGCTCTCT"  "X801_AGGACTTTCAGTGTTG" 
 [133] "X801_AGGATAATCATACGAC"  "X801_AGGATCTTCACTGCTC" 
 [135] "X801_AGGCATTCAGTGTATC"  "X801_AGGCATTGTCAGGCAA" 
 [137] "X801_AGGGAGTGTGTTCCAA"  "X801_AGGGTCCTCGGTCACG" 
 [139] "X801_AGGGTGAAGGGCCAAT"  "X801_AGGGTTTCATGTGGTT" 
 [141] "X801_

In [45]:
table(reference_data@meta.data[reference_data@meta.data$subgroup == "SHH",]$label)


             macrophage_monocytes oligodendrocytes_astrocytes_other 
                              432                                64 
                           SHH-A1                            SHH-A2 
                              789                               225 
                           SHH-B1                            SHH-B2 
                             1513                               770 
                           SHH-C1                            SHH-C2 
                             1682                               588 
                           SHH-X1 
                               12 

In [10]:
table(reference_data@meta.data$Subpopulations)

< table of extent 0 >

In [8]:
paste0(base.path, "/SHH_Riemondy/")

[1] "/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/Working_project/MB/scRNA_reference/SHH_Riemondy/"